In [21]:
import os
import sys
import pandas as pd
import numpy as np
import glob
import time
import gget
import scipy
from scipy.sparse import csr_matrix
import anndata as an
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
import random
from importlib import reload
import warnings
import ot
from scipy.spatial.distance import pdist, squareform
from matplotlib.colors import ListedColormap
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import MinMaxScaler

"""WARNING: no warnings"""
warnings.filterwarnings("ignore")

source_path = os.path.abspath("../utilities/calculations/")
sys.path.append(source_path)
import centrality as central

## Load Population Data

In [22]:
%%time 
# 1MB resolution
resolution = 1000000

fpath = f"/nfs/turbo/umms-indikar/shared/projects/poreC/pipeline_outputs/higher_order/anndata/population_mESC_{resolution}_features.h5ad"

adata = sc.read_h5ad(fpath)

sc.logging.print_memory_usage()

adata

Memory usage: current 5.37 GB, difference +2.40 GB
CPU times: user 12.3 s, sys: 2.8 s, total: 15.1 s
Wall time: 20 s


AnnData object with n_obs × n_vars = 2579 × 2756467
    obs: 'bin_index', 'bin_start', 'bin_end', 'bin', 'chrom', 'chrom_bin', 'degree', 'genes', 'n_genes', 'ATACSeq_1', 'ATACSeq_2', 'ATACSeq_3', 'CTCF', 'H3K27ac', 'H3K27me3', 'RNA_1', 'RNA_2', 'RNA_3', 'RNA_4', 'RNA_5', 'RNA_6', 'PolII'
    var: 'read_index', 'basename', 'mean_mapq', 'median_mapq', 'n_chromosomes', 'order', 'n_bins', 'read_length_bp', 'genes', 'n_genes'
    uns: 'base_resolution', 'chrom_sizes', 'gdf', 'gene_map', 'intervals'
    layers: 'H'

In [23]:
adata.obs["bin_index"]

bin_name
chr9:121        0
chr19:26        1
chr4:127        2
chr8:21         3
chr10:57        4
             ... 
chr16:7      2574
chr1:159     2575
chr9:116     2576
chr14:125    2577
chr1:195     2578
Name: bin_index, Length: 2579, dtype: int64

In [24]:
# filter
H = adata.X.tocsr().astype(float)
print(f"raw shape = {H.shape}")

raw shape = (2579, 2756467)


In [25]:
# IQR filter on locus degree (rows)
s = np.asarray(H.sum(axis=1)).ravel()
q1, q3 = np.quantile(s, [0.25, 0.75])
iqr = q3 - q1
low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
row_mask = (s >= low) & (s <= high)

H = H[row_mask, :]
obs_idx = np.where(row_mask)[0]          # maps H rows -> adata.obs rows

In [26]:
# drop singletons (reads with < 2 loci after row filter)
col_sums = np.asarray(H.sum(axis=0)).ravel()
col_mask = col_sums >= 2
H = H[:, col_mask]
var_idx = np.where(col_mask)[0]          # maps H cols -> adata.var cols

In [27]:
# drop large hyperedges (reads with > 10 loci)
col_sums = np.asarray(H.sum(axis=0)).ravel()
col_mask = col_sums <= 10
H = H[:, col_mask]
var_idx = var_idx[col_mask]

print(f"filtered shape = {H.shape}")

filtered shape = (2335, 2039865)


In [29]:
# centrality
node_scores, edge_scores = central.hevc(
    H,
    function='log-exp',   # 'linear', 'log-exp', or 'max'
    maxiter=10000,
    tol=1e-6,
    verbose=True
)
print("done!")

1 ...
2 ...
3 ...
4 ...
5 ...
6 ...
7 ...
8 ...
9 ...
25 ===
done!


In [37]:
adata.obs

,bin_index,bin_start,bin_end,bin,chrom,chrom_bin,degree,genes,n_genes,ATACSeq_1,...,CTCF,H3K27ac,H3K27me3,RNA_1,RNA_2,RNA_3,RNA_4,RNA_5,RNA_6,PolII
bin_name,,,,,,,,,,,,,,,,,,,,,
chr9:121,0,121000000,122000000,1394,9,121,3532,Gm47092;Higd1a;Gm47108;Gask1a;Lyzl4os;Gm47112;...,41,0.826484,...,1.149226,1.349552,0.866066,0.295370,0.387013,0.162437,0.225783,0.573875,0.122417,1.016765
chr19:26,1,26000000,27000000,2436,19,26,3346,Gm50378;Smarca2;Gm50376;Gm48775;Gm815;1700048O...,11,0.497386,...,0.547185,0.336787,0.839273,0.133999,0.148066,0.132291,0.094680,0.141617,0.084908,0.417178
chr4:127,2,127000000,128000000,665,4,127,3768,Gjb3;Gm22221;Gm12943;Zmym6;Tmem35b;Gjb5;Gm1294...,16,0.754788,...,1.027046,1.577616,0.839461,0.521087,0.998980,0.293742,0.247401,0.906364,0.256514,0.750054
chr8:21,3,21000000,22000000,1163,8,21,12931,Defa5;Gm23142;Gm21092;Gm15304;Gm56829;Gm6689;D...,47,0.226766,...,0.218299,0.135160,0.243308,0.130106,0.336687,0.121854,0.090845,0.063376,0.155842,0.187819
chr10:57,4,57000000,58000000,1455,10,57,3285,Serinc1;Smpdl3a;Rpl48-ps1;Gm19256;Gm48055;Fabp...,14,0.472903,...,0.383735,0.229388,0.600799,0.238078,0.560188,0.233549,0.215554,0.490113,0.182086,0.176728
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
chr16:7,2574,7000000,8000000,2131,16,7,3437,Gm49535;Rbfox1;Gm49534;Gm49533,4,0.475243,...,0.438346,0.469556,0.659370,0.148148,0.068007,0.179972,0.094459,0.056656,0.136702,0.305652
chr1:159,2575,159000000,160000000,159,1,159,3435,Gm24719;Tnn;Tnr;Gm37757;Gm37731;Cop1;Gm38039;G...,15,0.752434,...,0.549626,0.373921,0.752304,0.179105,0.177992,0.340071,0.134395,0.253093,0.089073,0.572138
chr9:116,2576,116000000,117000000,1389,9,116,3541,Rbms3;Gm31410;Gm48038;D730003K21Rik;Tgfbr2;Gm4668,6,0.732873,...,0.708854,0.643508,0.690798,0.118125,0.252528,0.170411,0.084363,0.289917,0.141768,0.441737


In [39]:
node_df = pd.DataFrame({
    #"bin_name":                       ("chr" + adata.obs["chrom"].iloc[obs_idx].astype(str) + ":" +
    #                                   adata.obs["chrom_bin"].iloc[obs_idx].astype(str)).values,
    "bin_name":                       adata.obs.iloc[obs_idx].index,
    "chrom":                          adata.obs["chrom"].iloc[obs_idx].values,
    "bin_start":                      adata.obs["bin_start"].iloc[obs_idx].values,
    "bin_end":                        adata.obs["bin_end"].iloc[obs_idx].values,
    "global_hge_logexp_unweighted":   node_scores,
})

edge_df = pd.DataFrame({
    "read_name":                      adata.var.iloc[var_idx].index,
    "read_index":                     adata.var["read_index"].iloc[var_idx].values,
    "n_bins":       adata.var["n_bins"].iloc[var_idx].values,
    "global_hge_logexp_unweighted":   edge_scores,
})

print(node_df.head())
print(edge_df.head())

   bin_name chrom  bin_start    bin_end  global_hge_logexp_unweighted
0  chr9:121     9  121000000  122000000                      0.000428
1  chr19:26    19   26000000   27000000                      0.000421
2  chr4:127     4  127000000  128000000                      0.000433
3  chr10:57    10   57000000   58000000                      0.000425
4   chr12:8    12    8000000    9000000                      0.000426
                              read_name  read_index  n_bins  \
0  3891ee6d-53d1-4ee0-ba2f-3d22291d4493           0       2   
1  66953ddf-e76d-4cdf-aaf8-be028a2d7b04           1      12   
2  ad5b2240-893f-4ed0-a157-c2be66d8d754           2       7   
3  207b28dd-a1ae-443a-9c8e-d09a2dd018dd           5       2   
4  3f354c45-5e48-4f6d-8c7e-05369432b344           6       2   

   global_hge_logexp_unweighted  
0                  1.346761e-06  
1                  3.508383e-30  
2                  1.994061e-23  
3                  1.337412e-06  
4                  1.293423e-06

In [40]:
print("=== Node scores ===")
print(node_df["global_hge_logexp_unweighted"].describe())
print(f"  zeros:     {(node_df['global_hge_logexp_unweighted'] == 0).sum()}")
print(f"  neg:       {(node_df['global_hge_logexp_unweighted'] < 0).sum()}")

print("\n=== Edge scores ===")
print(edge_df["global_hge_logexp_unweighted"].describe())
print(f"  zeros:     {(edge_df['global_hge_logexp_unweighted'] == 0).sum()}")
print(f"  neg:       {(edge_df['global_hge_logexp_unweighted'] < 0).sum()}")

=== Node scores ===
count    2335.000000
mean        0.000428
std         0.000005
min         0.000408
25%         0.000425
50%         0.000429
75%         0.000431
max         0.000446
Name: global_hge_logexp_unweighted, dtype: float64
  zeros:     0
  neg:       0

=== Edge scores ===
count    2.039865e+06
mean     4.902285e-07
std      6.443969e-07
min      1.228231e-33
25%      2.399445e-13
50%      5.722346e-10
75%      1.326836e-06
max      1.441523e-06
Name: global_hge_logexp_unweighted, dtype: float64
  zeros:     0
  neg:       0


In [41]:
spath = "/nfs/turbo/umms-indikar/shared/projects/poreC/pipeline_outputs/higher_order/global_core_score/population_mESC_1000000_scores.csv"

score = pd.read_csv(spath)
score.columns

Index(['bin_name', 'bin_index', 'bin_start', 'bin_end', 'bin', 'chrom',
       'chrom_bin', 'degree', 'genes', 'n_genes', 'ATACSeq_1', 'ATACSeq_2',
       'ATACSeq_3', 'CTCF', 'H3K27ac', 'H3K27me3', 'RNA_1', 'RNA_2', 'RNA_3',
       'RNA_4', 'RNA_5', 'RNA_6', 'degree_outlier', 'chrom_degree',
       'ce_singular_vector_1', 'ce_eigenvector_centrality',
       'ce_betweenness_centrality', 'ce_pagerank', 'hge_singular_vector_1',
       'hge_logexp_unweighted', 'hge_logexp_degree_weighted',
       'hge_logexp_RNA_weighted', 'hge_logexp_ATAC_weighted',
       'global_singular_vector_1', 'global_hge_logexp_unweighted',
       'global_hge_logexp_RNA_weighted'],
      dtype='object')

In [42]:
node_df.to_csv("/nfs/turbo/umms-indikar/shared/projects/poreC/pipeline_outputs/higher_order/global_core_score/pop_mESC_1000000_hge_logexp_nodes.csv", index=False)


edge_df.to_csv("/nfs/turbo/umms-indikar/shared/projects/poreC/pipeline_outputs/higher_order/global_core_score/pop_mESC_1000000_hge_logexp_edges.csv", index=False)




In [ ]:
node_df.describe()

In [ ]:
threshold = node_df['global_hge_logexp_unweighted'].quantile(0.89)
kneedle_df = node_df[node_df['global_hge_logexp_unweighted'] >= threshold]

In [ ]:
kneedle_df.describe()

In [ ]:
import matplotlib.pyplot as plt

p25 = node_df["global_hge_logexp_unweighted"].quantile(0.90)

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(node_df["global_hge_logexp_unweighted"], bins=100, color="#46ACC8", edgecolor="none")
ax.axvline(p25, color="#B40F20", linestyle="--", label=f"top 10% threshold")
ax.set_xlabel("Hypergraph Eigenvector Centrality (Log Exp)")
ax.set_ylabel("count")
#ax.set_title("Distribution of Node Centrality Scores")
ax.legend(frameon=False)
sns.despine(top=True)
plt.tight_layout()
plt.show()